# lets_route — `route_f` Playground

This notebook runs on the **`route_f`** kernel (Python 3.12, uv-managed).
The same environment is used in the terminal via:

```bash
source .route_f/bin/activate
```

> If the kernel isn't listed in VS Code yet, finish the setup first:
> `./setup_route_f.sh` (or verify with `jupyter kernelspec list`).

In [1]:
import sys, platform
print(f"Python : {sys.version.split()[0]} (executable: {sys.executable})")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"torch : {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU : {torch.cuda.get_device_name(0)}")
    try:
        torch.zeros(1, device=torch.cuda.is_available() and 'cuda' or 'cpu')
        print("tensor allocation ok")
    except Exception as e:
        print("tensor allocation failed:", e)
except ImportError:
    print("torch not installed yet — run ./setup_route_f.sh")

Python : 3.12.13 (executable: /home/mr/Desktop/Fireworks/.route_f/bin/python)
Platform: Linux-7.0.0-38-generic-x86_64-with-glibc2.39
torch : 2.15.0+cu132
CUDA available : True
GPU : NVIDIA GeForce RTX 3060 Laptop GPU
tensor allocation ok


In [2]:
# Quick sanity check of the training stack
import importlib
mods = ["transformers", "datasets", "accelerate", "peft", "pandas", "numpy", "pyarrow"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:16s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{m:16s} NOT INSTALLED")

/home/mr/Desktop/Fireworks/.route_f/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers     5.22.1
datasets         5.1.0
accelerate       1.15.0
peft             0.21.2
pandas           3.1.0
numpy            2.5.4
pyarrow          26.0.0


## Dataset: xRouteBench (`ulab-ai/xRouteBench`)

LLM-routing benchmark — each row is a `(query, candidate model)` pair with the
model's response, token usage, latency, and a `performance` score. A router
learns which model to pick per query, trading off performance vs price.

- **Configs (scenarios):** `llmrouter_generic`, `memory_locomo`, `memory_longmemeval`,
  `timeseries`, `video`, `multimodal_geometry3k`, `multimodal_mathvista`, `personalized`.
- **Raw queries:** `<scenario>_queries` (train/valid/test) — no model executions.
- **Pricing pool:** `llm_candidates` (18 models with prices).
- **No bulk download:** the cells below inspect everything via the HF API and
  stream only a few rows (parquet pulled lazily). Set upstream=False`` only when
  you actually want to download the full dataset for training.

In [3]:
import sys
sys.path.insert(0, ".")
from data.load_xroutebench import list_configs, schema, peek

print("Available configs (from HF datasets-server API, no download):")
for c in list_configs():
    print("  -", c)

cols, samples = schema("llmrouter_generic", "train")
print("\nSchema for llmrouter_generic/train:")
print(cols)

print("\nFirst 2 rows via STREAMING (only the first parquet chunk is read):")
for row in peek("llmrouter_generic", "train", n=2):
    print("  -", {k: row[k] for k in ("task_name", "model_name", "metric")})

Available configs (from HF datasets-server API, no download):
  - llm_candidates
  - llmrouter_generic
  - llmrouter_generic_queries
  - memory_locomo
  - memory_locomo_queries
  - memory_longmemeval
  - memory_longmemeval_queries
  - multimodal_geometry3k
  - multimodal_geometry3k_queries
  - multimodal_mathvista
  - multimodal_mathvista_queries
  - personalized
  - personalized_queries
  - timeseries
  - timeseries_queries
  - video
  - video_queries

Schema for llmrouter_generic/train:
['task_name', 'query', 'ground_truth', 'metric', 'choices', 'task_id', 'model_name', 'response', 'token_num', 'input_tokens', 'output_tokens', 'response_time', 'performance', 'embedding_id']

First 2 rows via STREAMING (only the first parquet chunk is read):
  - {'task_name': 'mbpp', 'model_name': 'qwen2.5-7b-instruct', 'metric': 'code_eval'}
  - {'task_name': 'mbpp', 'model_name': 'gemma-2-9b-it', 'metric': 'code_eval'}


In [ ]:
# Download the pinned official Qwen3.5-0.8B checkpoint into this repo.
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "route_f_playground.ipynb").exists()
)
MODEL_DIR = repo_root / "models" / "qwen3.5-0.8b"

snapshot_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    local_dir=MODEL_DIR,
)
downloaded_bytes = sum(
    path.stat().st_size for path in MODEL_DIR.rglob("*") if path.is_file()
)
print(f"Downloaded {MODEL_ID}@{MODEL_REVISION[:8]}")
print(f"Location: {snapshot_path}")
print(f"Size: {downloaded_bytes / 1024**3:.2f} GiB")

In [5]:
# Read one real xRouteBench Parquet query and run it through local Qwen.
import pandas as pd
import torch
from transformers import AutoModelForMultimodalLM, AutoTokenizer

SAMPLE_TASK = "mmlu"
MAX_NEW_TOKENS = 128
parquet_path = repo_root / "data/raw/llmrouter_generic/test-000.parquet"
sample = (
    pd.read_parquet(
        parquet_path,
        columns=["embedding_id", "task_name", "query", "ground_truth"],
    )
    .query("task_name == @SAMPLE_TASK")
    .sort_values("embedding_id")
    .drop_duplicates("embedding_id")
    .iloc[0]
)

# xRouteBench stores the system and user portions inside one query string.
raw_query = str(sample["query"])
system_marker, user_marker = "[System Instruction]", "[User Query]"
if raw_query.startswith(system_marker) and user_marker in raw_query:
    system_text, user_text = raw_query[len(system_marker):].split(user_marker, 1)
    messages = [
        {"role": "system", "content": system_text.strip()},
        {"role": "user", "content": user_text.strip()},
    ]
else:
    messages = [{"role": "user", "content": raw_query}]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    model_dtype = torch.float32

qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
qwen_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_DIR, local_files_only=True, dtype=model_dtype
).to(device).eval()
formatted_prompt = qwen_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = qwen_tokenizer(formatted_prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    generated = qwen_model.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
response_tokens = generated[0, model_inputs["input_ids"].shape[-1]:]
qwen_response = qwen_tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

print(f"Task: {sample['task_name']} | embedding_id: {sample['embedding_id']}")
print(f"Ground truth: {sample['ground_truth']}")
print("\nDataset query:\n", raw_query)
print("\nQwen response:\n", qwen_response)

Task: mmlu | embedding_id: 14
Ground truth: A

Dataset query:
 [System Instruction]
Answer the following multiple-choice question by selecting the correct option (A, B, C, or D). You MUST put your final answer letter in a parenthesis.

[User Query]
## Question:
The ecological approach to urban sociology involved studying:

## Options:
A. how social groups colonized different areas of the city and competed for resources
B. the forms of wildlife and natural habitats that could be found on the edges of the city
C. the way in which people organized collective protests about environmental issues
D. how men and women used the city's public spaces differently

Qwen response:
 A


## Plug-and-play LLM evaluation on xRouteBench Parquet

The loader below validates the full query-by-model matrix, returns one row per query instance, and keeps ground truth separate from the messages sent to an LLM. The evaluation harness accepts any generator that returns the same small result dictionary, so local checkpoints and API adapters can be swapped without changing sampling or scoring.

This first check uses 12 deterministic multiple-choice queries because their `em_mc` ground truth can be scored exactly. Dollar cost for local Qwen is reported only when a real hardware cost per hour is supplied; latency and token throughput are always measured.

In [ ]:
import hashlib
import re
import time
import numpy as np
import pandas as pd

ROUTING_COLUMNS = [
    "embedding_id", "task_name", "query", "ground_truth",
    "metric", "model_name", "performance", "input_tokens",
    "output_tokens", "response_time",
]

def load_xroute_queries(repo_root, split="test"):
    """Return one validated row per query and the complete long matrix."""
    path = Path(repo_root) / f"data/raw/llmrouter_generic/{split}-000.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing xRouteBench Parquet: {path}")
    frame = pd.read_parquet(path, columns=ROUTING_COLUMNS)
    missing = set(ROUTING_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if frame.duplicated(["embedding_id", "model_name"]).any():
        raise ValueError("Duplicate (embedding_id, model_name) executions found")

    candidate_count = frame["model_name"].nunique()
    per_query = frame.groupby("embedding_id", sort=False).agg(
        rows=("model_name", "size"),
        models=("model_name", "nunique"),
        queries=("query", "nunique"),
        tasks=("task_name", "nunique"),
        metrics=("metric", "nunique"),
    )
    invalid = per_query[
        (per_query["rows"] != candidate_count)
        | (per_query["models"] != candidate_count)
        | (per_query[["queries", "tasks", "metrics"]] != 1).any(axis=1)
    ]
    if len(invalid):
        raise ValueError(f"{len(invalid)} incomplete or inconsistent query matrices")

    query_table = (
        frame.sort_values("embedding_id")
        .drop_duplicates("embedding_id")
        [["embedding_id", "task_name", "query", "ground_truth", "metric"]]
        .reset_index(drop=True)
    )
    query_table["query_hash"] = query_table["query"].map(
        lambda value: hashlib.sha256(str(value).encode("utf-8")).hexdigest()
    )
    audit = {
        "split": split,
        "execution_rows": len(frame),
        "query_instances": len(query_table),
        "unique_prompt_texts": query_table["query_hash"].nunique(),
        "candidate_models": candidate_count,
    }
    return query_table, frame, audit

def select_mc_queries(query_table, tasks, per_task=2, seed=42):
    """Deterministically sample exact-score multiple-choice queries by task."""
    selections = []
    for task in tasks:
        task_rows = query_table[
            query_table["task_name"].eq(task) & query_table["metric"].eq("em_mc")
        ].copy()
        if len(task_rows) < per_task:
            raise ValueError(f"Task {task!r} has only {len(task_rows)} em_mc queries")
        task_rows["sample_order"] = task_rows["query_hash"].map(
            lambda value: hashlib.sha256(f"{seed}:{value}".encode()).hexdigest()
        )
        selections.append(task_rows.sort_values("sample_order").head(per_task))
    selected = pd.concat(selections, ignore_index=True).drop(columns="sample_order")
    if len(selected) < 10:
        raise ValueError("Evaluation must contain at least 10 queries")
    return selected

def xroute_messages(raw_query):
    """Convert xRouteBench's combined prompt into proper chat roles."""
    raw_query = str(raw_query)
    system_marker, user_marker = "[System Instruction]", "[User Query]"
    if raw_query.startswith(system_marker) and user_marker in raw_query:
        system_text, user_text = raw_query[len(system_marker):].split(user_marker, 1)
        return [
            {"role": "system", "content": system_text.strip()},
            {"role": "user", "content": user_text.strip()},
        ]
    return [{"role": "user", "content": raw_query}]

def extract_mc_choice(response):
    """Extract the final A-D choice without rewarding letters in prose."""
    text = str(response).strip().upper()
    parenthesized = re.findall(r"\(([A-E])\)", text)
    if parenthesized:
        return parenthesized[-1]
    final_answer = re.findall(r"(?:FINAL\s+)?ANSWER(?:\s+IS)?[:\s]+([A-E])(?:\b|$)", text)
    if final_answer:
        return final_answer[-1]
    leading = re.match(r"^\s*([A-E])(?:[.)]|\s*[-:])", text)
    if leading:
        return leading.group(1)
    standalone = re.fullmatch(r"\s*([A-E])(?:[.)])?\s*", text)
    return standalone.group(1) if standalone else None

def evaluate_generator(queries, generate_fn):
    """Evaluate any adapter returning response/tokens/latency/cost fields."""
    rows = []
    for position, row in enumerate(queries.itertuples(index=False), start=1):
        try:
            result = generate_fn(xroute_messages(row.query))
            parsed = extract_mc_choice(result["response"])
            error = None
        except Exception as exc:
            result = {"response": "", "input_tokens": 0, "output_tokens": 0, "latency_s": np.nan, "cost_usd": np.nan}
            parsed, error = None, f"{type(exc).__name__}: {exc}"
        expected = str(row.ground_truth).strip().upper()
        rows.append({
            "embedding_id": int(row.embedding_id),
            "task_name": row.task_name,
            "expected": expected,
            "parsed_answer": parsed,
            "correct": bool(parsed == expected),
            "response": result.get("response", ""),
            "input_tokens": result.get("input_tokens", np.nan),
            "output_tokens": result.get("output_tokens", np.nan),
            "latency_s": result.get("latency_s", np.nan),
            "cost_usd": result.get("cost_usd", np.nan),
            "finish_reason": result.get("finish_reason"),
            "error": error,
        })
        print(f"[{position:02d}/{len(queries)}] {row.task_name}: expected={expected}, predicted={parsed}")
    return pd.DataFrame(rows)

In [ ]:
EVAL_TASKS = ("mmlu", "arc_challenge", "openbook_qa", "commonsense_qa", "boolq", "hellaswag")
query_table, test_long, loader_audit = load_xroute_queries(repo_root, split="test")
eval_queries = select_mc_queries(query_table, EVAL_TASKS, per_task=2, seed=42)
print(loader_audit)
print("\nEvaluation sample:")
print(eval_queries[["embedding_id", "task_name", "ground_truth"]].to_string(index=False))

In [ ]:
# Set this to your real amortized machine/GPU cost to obtain local dollar cost.
LOCAL_COMPUTE_USD_PER_HOUR = None
EVAL_MAX_NEW_TOKENS = 128

def generate_with_local_qwen(messages):
    start = time.perf_counter()
    prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    with torch.inference_mode():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=EVAL_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()
    latency_s = time.perf_counter() - start
    response_ids = output[0, inputs["input_ids"].shape[-1]:]
    response = qwen_tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    finish_reason = (
        "eos" if response_ids.numel() and response_ids[-1].item() == qwen_tokenizer.eos_token_id
        else "length"
    )
    cost_usd = (
        latency_s * LOCAL_COMPUTE_USD_PER_HOUR / 3600.0
        if LOCAL_COMPUTE_USD_PER_HOUR is not None else np.nan
    )
    return {
        "response": response,
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "output_tokens": int(response_ids.numel()),
        "latency_s": latency_s,
        "cost_usd": cost_usd,
        "finish_reason": finish_reason,
    }

qwen_eval = evaluate_generator(eval_queries, generate_with_local_qwen)
qwen_accuracy = qwen_eval["correct"].mean()
qwen_tokens_per_s = qwen_eval["output_tokens"].sum() / qwen_eval["latency_s"].sum()
print("\nPer-query results:")
print(qwen_eval[["embedding_id", "task_name", "expected", "parsed_answer", "correct", "finish_reason", "latency_s", "response"]].to_string(index=False))
print(f"\nQwen accuracy: {qwen_accuracy:.1%} ({qwen_eval['correct'].sum()}/{len(qwen_eval)})")
print(f"Mean latency: {qwen_eval['latency_s'].mean():.3f}s | Output throughput: {qwen_tokens_per_s:.2f} tokens/s")
print(f"Length-truncated responses: {(qwen_eval['finish_reason'] == 'length').sum()}/{len(qwen_eval)}")
if LOCAL_COMPUTE_USD_PER_HOUR is None:
    print("Dollar cost: not estimated—set LOCAL_COMPUTE_USD_PER_HOUR to your real amortized hardware rate.")
else:
    print(f"Estimated local cost per 1,000 queries: ${qwen_eval['cost_usd'].mean() * 1000:.4f}")

In [ ]:
# Compare Qwen's measured accuracy with recorded xRouteBench candidates on the same IDs.
candidate_prices = pd.read_parquet(
    repo_root / "data/raw/llm_candidates/train-000.parquet"
)[["model_name", "input_price_per_1m", "output_price_per_1m"]]
selected_ids = set(eval_queries["embedding_id"])
candidate_eval = test_long[test_long["embedding_id"].isin(selected_ids)].merge(
    candidate_prices, on="model_name", how="left", validate="many_to_one"
)
candidate_eval["cost_usd"] = (
    candidate_eval["input_tokens"] * candidate_eval["input_price_per_1m"]
    + candidate_eval["output_tokens"] * candidate_eval["output_price_per_1m"]
) / 1_000_000.0
candidate_summary = (
    candidate_eval.groupby("model_name")
    .agg(
        accuracy=("performance", "mean"),
        mean_cost_per_1k_usd=("cost_usd", lambda values: values.mean() * 1000),
        mean_latency_s=("response_time", "mean"),
    )
    .reset_index()
)
qwen_summary = pd.DataFrame([{
    "model_name": "Qwen3.5-0.8B (local)",
    "accuracy": qwen_accuracy,
    "mean_cost_per_1k_usd": qwen_eval["cost_usd"].mean() * 1000,
    "mean_latency_s": qwen_eval["latency_s"].mean(),
}])
comparison = pd.concat([qwen_summary, candidate_summary], ignore_index=True)
comparison = comparison.sort_values(["accuracy", "mean_cost_per_1k_usd"], ascending=[False, True])
print(comparison.to_string(index=False, na_rep="not configured", float_format=lambda value: f"{value:.6f}"))